# Smart Traffic Light Traffic-Police DQN Training (Kaggle)

This notebook trains a traffic-police-style controller from the `dev-truong` branch of `truongNgn/smart-traffic-light-system`.

The action space matches a real traffic officer at a Vietnamese four-way intersection:

- `Action 0 = EAST_WEST`: both opposite directions on the East-West road get green together.
- `Action 1 = NORTH_SOUTH`: both opposite directions on the North-South road get green together.

The environment also uses a hard safety guard: if a phase is red too long, it is forced next. This prevents the high-traffic model from starving the quieter road. Training uses a curriculum: normal traffic, balanced heavy traffic, East-West peak traffic, and North-South peak traffic. The final cell creates `dqn_eval_best.pt`, selected by evaluation across all scenarios.


## 1. Install SUMO and clone the repo

In [ ]:
!pip install -q eclipse-sumo traci sumolib libsumo

In [ ]:
!git clone --branch dev-truong --single-branch https://github.com/truongNgn/smart-traffic-light-system.git
%cd smart-traffic-light-system


In [ ]:
# torch is already preinstalled on Kaggle's GPU image - don't reinstall it
!pip install -q pydantic pydantic-settings structlog gymnasium numpy tqdm

## 1.1. Verify 2-phase traffic-police logic

`dev-truong` is the default branch for this project. This cell verifies that Kaggle cloned a commit with the improved 2-phase action space before spending time training.


In [ ]:
# Verify/patch the cloned dev-truong code for 2-phase action space and max-red starvation guard.
# This keeps the Kaggle notebook usable even if dev-truong has not been pushed with the latest local guard code yet.
from pathlib import Path


def replace_once(path, old, new):
    path = Path(path)
    text = path.read_text(encoding="utf-8")
    if new in text:
        return
    if old not in text:
        raise RuntimeError(f"Patch target not found in {path}: {old[:120]!r}")
    path.write_text(text.replace(old, new, 1), encoding="utf-8")


env_path = Path("rl/env/traffic_env.py")
env_text = env_path.read_text(encoding="utf-8")
if "max_red_time_s" not in env_text:
    replace_once(
        env_path,
        "        green_duration_s: float = GREEN_DURATION_S,\n        initial_phase: PhaseAction = PhaseAction.EAST_WEST,",
        "        green_duration_s: float = GREEN_DURATION_S,\n        max_red_time_s: float | None = 90.0,\n        initial_phase: PhaseAction = PhaseAction.EAST_WEST,",
    )
    replace_once(
        env_path,
        "        self.green_duration_s = green_duration_s\n        self.initial_phase = initial_phase",
        "        self.green_duration_s = green_duration_s\n        self.max_red_time_s = max_red_time_s\n        self.initial_phase = initial_phase",
    )
    replace_once(
        env_path,
        "        self._current_phase: PhaseAction | None = None\n        self._cumulative_arrived = 0",
        "        self._current_phase: PhaseAction | None = None\n        self._red_time_by_phase: dict[PhaseAction, float] = {phase: 0.0 for phase in PhaseAction}\n        self._last_action_was_forced = False\n        self._cumulative_arrived = 0",
    )
    replace_once(
        env_path,
        "        self._tls = TlsController(self._sim.traci, self.tls_id)\n        self._cumulative_arrived = 0",
        "        self._tls = TlsController(self._sim.traci, self.tls_id)\n        self._cumulative_arrived = 0\n        self._red_time_by_phase = {phase: 0.0 for phase in PhaseAction}\n        self._last_action_was_forced = False",
    )
    replace_once(
        env_path,
        "        self._current_phase = self.initial_phase\n        self._step_and_track()",
        "        self._current_phase = self.initial_phase\n        self._step_and_track(green_phase=self.initial_phase)",
    )
    replace_once(
        env_path,
        "        phase = PhaseAction(action)\n        if phase != self._current_phase:",
        "        requested_phase = PhaseAction(action)\n        phase = self._phase_after_safety_guard(requested_phase)\n        if phase != self._current_phase:",
    )
    replace_once(
        env_path,
        "        else:\n            self._step_and_track(self._green_steps)",
        "        else:\n            self._step_and_track(self._green_steps, green_phase=phase)",
    )
    replace_once(
        env_path,
        '            "phase": phase.name,\n        }\n        return obs, reward, terminated, truncated, info\n\n    def _apply_phase_transition',
        '            "phase": phase.name,\n            "requested_phase": requested_phase.name,\n            "action_forced_by_guard": self._last_action_was_forced,\n            "red_time_s": {phase.name: red_time for phase, red_time in self._red_time_by_phase.items()},\n        }\n        return obs, reward, terminated, truncated, info\n\n    def _phase_after_safety_guard(self, requested_phase: PhaseAction) -> PhaseAction:\n        self._last_action_was_forced = False\n        if self.max_red_time_s is None:\n            return requested_phase\n\n        starving_phases = [\n            phase\n            for phase, red_time_s in self._red_time_by_phase.items()\n            if phase != self._current_phase and red_time_s >= self.max_red_time_s\n        ]\n        if not starving_phases:\n            return requested_phase\n\n        self._last_action_was_forced = starving_phases[0] != requested_phase\n        return starving_phases[0]\n\n    def _apply_phase_transition',
    )
    replace_once(
        env_path,
        "        self._step_and_track(self._green_steps)\n\n    def _step_and_track(self, n: int = 1) -> None:",
        "        self._step_and_track(self._green_steps, green_phase=new_phase)\n\n    def _step_and_track(self, n: int = 1, *, green_phase: PhaseAction | None = None) -> None:",
    )
    replace_once(
        env_path,
        "        for _ in range(n):\n            self._sim.step(1)\n            self._cumulative_arrived += self._sim.traci.simulation.getArrivedNumber()",
        "        for _ in range(n):\n            self._sim.step(1)\n            self._update_red_times(green_phase)\n            self._cumulative_arrived += self._sim.traci.simulation.getArrivedNumber()\n\n    def _update_red_times(self, green_phase: PhaseAction | None) -> None:\n        for phase in PhaseAction:\n            if phase == green_phase:\n                self._red_time_by_phase[phase] = 0.0\n            else:\n                self._red_time_by_phase[phase] += self.step_length_s",
    )

config_path = Path("rl/train/config.py")
config_text = config_path.read_text(encoding="utf-8")
if "max_red_time_s" not in config_text:
    replace_once(
        config_path,
        '    green_duration_s: float = Field(\n        default=10.0,\n        gt=0.0,\n        description="Seconds to hold each selected green action; 10s matches the paper.",\n    )\n    backend: str = Field(',
        '    green_duration_s: float = Field(\n        default=10.0,\n        gt=0.0,\n        description="Seconds to hold each selected green action; 10s matches the paper.",\n    )\n    max_red_time_s: float | None = Field(\n        default=90.0,\n        description="Safety guard: force a phase if it has been red longer than this.",\n    )\n    backend: str = Field(',
    )

train_path = Path("rl/train/train.py")
train_text = train_path.read_text(encoding="utf-8")
if "max_red_time_s=cfg.max_red_time_s" not in train_text:
    replace_once(
        train_path,
        "        green_duration_s=cfg.green_duration_s,\n        backend=backend,",
        "        green_duration_s=cfg.green_duration_s,\n        max_red_time_s=cfg.max_red_time_s,\n        backend=backend,",
    )

from common.constants import DQN_OUTPUT_SIZE, NUM_ACTIONS, PHASE_DIRECTIONS, PhaseAction
from rl.env.traffic_env import SumoTrafficEnv
from rl.train.config import TrainingConfig

assert NUM_ACTIONS == 2, f"Expected 2 phase actions, got {NUM_ACTIONS}"
assert DQN_OUTPUT_SIZE == 2, f"Expected DQN output size 2, got {DQN_OUTPUT_SIZE}"
assert set(PhaseAction) == {PhaseAction.EAST_WEST, PhaseAction.NORTH_SOUTH}
assert len(PHASE_DIRECTIONS[PhaseAction.EAST_WEST]) == 2
assert len(PHASE_DIRECTIONS[PhaseAction.NORTH_SOUTH]) == 2
assert "initial_phase" in SumoTrafficEnv.__init__.__code__.co_varnames
assert "max_red_time_s" in SumoTrafficEnv.__init__.__code__.co_varnames
assert hasattr(TrainingConfig(), "max_red_time_s")
print("Verified: 2-phase actions + max-red starvation guard are available.")


In [ ]:
import os
import sumo

# eclipse-sumo bundles its own binaries + tools/ under the installed
# package directory - point SUMO_HOME there instead of a system path.
os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
print("SUMO_HOME =", os.environ["SUMO_HOME"])

!sumo --version

If `sumo --version` fails to print a version here, the `eclipse-sumo` wheel most likely didn't ship a binary for this exact platform. Fall back to the apt-get route as a last resort (slower, and known to segfault on some Kaggle images - see the note in cell 1):

```bash
!apt-get update -qq && apt-get install -y -qq sumo sumo-tools sumo-doc
```
```python
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"
```

## 2. Confirm GPU is visible to PyTorch, and libsumo is importable

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

import libsumo
print("libsumo importable OK - training will use the fast in-process backend")

## 3. Build the SUMO network and generate mixed demand scenarios

The previous model worked on normal traffic but failed under peak imbalance. This notebook generates four scenarios:

- `normal`: original demand.
- `heavy_2x`: all approaches doubled.
- `imbalanced_ew3x`: East-West peak road, North-South normal.
- `imbalanced_ns3x`: North-South peak road, East-West normal.


In [ ]:
!python -m simulation.net.build_net
!python -m simulation.net.generate_routes --duration 1800 --seed 42 --out simulation/net/intersection.rou.xml

from pathlib import Path
import xml.etree.ElementTree as ET


def make_scaled_scenario(name, *, duration_s=1800, all_scale=1.0, ew_scale=1.0, ns_scale=1.0):
    src = Path("simulation/net/intersection.rou.xml")
    route_path = Path(f"simulation/net/{name}.rou.xml")
    sumocfg_path = Path(f"simulation/net/{name}.sumocfg")
    tree = ET.parse(src)
    root = tree.getroot()

    for flow in root.findall("flow"):
        flow.set("end", str(duration_s))
        parts = flow.get("id", "").split("_")
        origin = parts[-2] if len(parts) >= 2 else ""
        axis_scale = ew_scale if origin in {"E", "W"} else ns_scale
        flow.set("vehsPerHour", f"{float(flow.get('vehsPerHour')) * all_scale * axis_scale:.2f}")

    ET.indent(tree, space="    ")
    tree.write(route_path, encoding="UTF-8", xml_declaration=True)
    sumocfg_path.write_text(f"""<?xml version="1.0" encoding="UTF-8"?>
<configuration>
    <input>
        <net-file value="intersection.net.xml"/>
        <route-files value="{route_path.name}"/>
        <additional-files value="vtypes.add.xml"/>
    </input>
    <time>
        <begin value="0"/>
        <step-length value="1"/>
    </time>
    <processing>
        <time-to-teleport value="-1"/>
    </processing>
    <report>
        <no-step-log value="true"/>
        <duration-log.disable value="true"/>
    </report>
</configuration>
""", encoding="utf-8")
    return str(sumocfg_path)


SCENARIOS = {
    "normal": "simulation/net/intersection.sumocfg",
    "heavy_2x": make_scaled_scenario("intersection_heavy_2x", all_scale=2.0),
    "imbalanced_ew3x": make_scaled_scenario("intersection_imbalanced_ew3x", ew_scale=3.0),
    "imbalanced_ns3x": make_scaled_scenario("intersection_imbalanced_ns3x", ns_scale=3.0),
}

SCENARIOS


## 4. Quick pipeline smoke test (optional but recommended)

Runs 2 tiny episodes end-to-end before committing to a long training run - catches setup problems in seconds instead of hours.

In [ ]:
from rl.train.config import TrainingConfig
from rl.train.train import train

smoke_cfg = TrainingConfig(
    num_episodes=2,
    episode_duration_s=60,
    checkpoint_dir="/kaggle/working/smoke_checkpoints",
    min_replay_size=8,
    batch_size=4,
)
_ = train(smoke_cfg)
print("Smoke test OK")

## 5. Train the improved traffic-police policy

This is curriculum training. It starts on normal traffic, then resumes the same model through heavier and imbalanced peak-hour scenarios. The max-red guard is enabled during training so the learned policy is optimized inside the same safety envelope used during evaluation.

The raw `dqn_best.pt` is still saved during training, but do **not** blindly deploy it. The next section evaluates candidate checkpoints across all scenarios and writes `dqn_eval_best.pt`.


In [ ]:
from pathlib import Path

from rl.train.config import TrainingConfig
from rl.train.train import train

CHECKPOINT_DIR = "/kaggle/working/checkpoints"


def train_stage(name, *, sumocfg_path, num_episodes, resume_from=None):
    print(f"\n=== Training stage: {name} -> episode {num_episodes} ===")
    cfg = TrainingConfig(
        sumocfg_path=sumocfg_path,
        num_episodes=num_episodes,
        episode_duration_s=1800,
        green_duration_s=10.0,
        max_red_time_s=90.0,
        checkpoint_dir=CHECKPOINT_DIR,
        resume_from=resume_from,
        batch_size=128,
        min_replay_size=5000,
        replay_capacity=100_000,
        learning_rate=5e-5,
        gamma=0.99,
        epsilon_start=1.0,
        epsilon_end=0.02,
        epsilon_decay_episodes=1400,
        target_sync_every_episodes=5,
        checkpoint_every_episodes=50,
        train_every_n_steps=4,
        log_every_episodes=10,
    )
    return train(cfg)


agent = train_stage("normal", sumocfg_path=SCENARIOS["normal"], num_episodes=500)
agent = train_stage(
    "heavy_2x",
    sumocfg_path=SCENARIOS["heavy_2x"],
    num_episodes=900,
    resume_from=f"{CHECKPOINT_DIR}/dqn_final.pt",
)
agent = train_stage(
    "imbalanced_ew3x",
    sumocfg_path=SCENARIOS["imbalanced_ew3x"],
    num_episodes=1200,
    resume_from=f"{CHECKPOINT_DIR}/dqn_final.pt",
)
agent = train_stage(
    "imbalanced_ns3x",
    sumocfg_path=SCENARIOS["imbalanced_ns3x"],
    num_episodes=1500,
    resume_from=f"{CHECKPOINT_DIR}/dqn_final.pt",
)


## 6. Resuming (only needed if a session got cut off)

Kaggle sessions get killed after ~9-12 hours. `/kaggle/working/` output persists between sessions, so if training didn't finish, start a new session and continue from the last checkpoint instead of restarting from scratch.

In [ ]:
# from rl.train.config import TrainingConfig
# from rl.train.train import train
#
# resumed_cfg = TrainingConfig(
#     num_episodes=1000,
#     episode_duration_s=3600,
#     checkpoint_dir="/kaggle/working/checkpoints",
#     resume_from="/kaggle/working/checkpoints/dqn_final.pt",
# )
# agent = train(resumed_cfg)

## 7. Evaluate checkpoints and select the deployment model

This evaluates recent periodic checkpoints plus `dqn_best.pt` and `dqn_final.pt` on held-out seeds. The selected checkpoint is copied to `/kaggle/working/checkpoints/dqn_eval_best.pt`. Lower score is better; the score favors lower mean/final waiting time, lower queue length, and higher throughput.

In [ ]:
from pathlib import Path
import shutil

from benchmark.policies import DQNPolicy, FixedTimePolicy
from benchmark.run_episode import run_episode
from rl.agent.dqn_agent import DQNAgent
from rl.env.traffic_env import SumoTrafficEnv
from rl.train.checkpoint import load_checkpoint

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
EVAL_DURATION_S = 1800
EVAL_SEEDS = [101, 102, 103]

periodic = sorted(CHECKPOINT_DIR.glob("dqn_episode_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
candidate_paths = []
for path in [CHECKPOINT_DIR / "dqn_best.pt", CHECKPOINT_DIR / "dqn_final.pt", *periodic[-12:]]:
    if path.exists() and path not in candidate_paths:
        candidate_paths.append(path)


def mean(values):
    return sum(values) / len(values) if values else 0.0


def aggregate(metrics):
    dicts = [m.to_dict() for m in metrics]
    return {key: mean([d[key] for d in dicts]) for key in dicts[0]}


def score(metrics):
    # Lower is better. Strongly penalize average/final waiting and queue;
    # reward throughput, but not enough to justify starving a phase.
    return (
        2.0 * metrics["mean_waiting_time_s"]
        + 1.0 * metrics["final_waiting_time_s"]
        + 30.0 * metrics["mean_queue_length"]
        - 1.0 * metrics["arrived_vehicles"]
    )


def eval_policy(sumocfg_path, policy, seed):
    env = SumoTrafficEnv(
        sumocfg_path=sumocfg_path,
        episode_duration_s=EVAL_DURATION_S,
        green_duration_s=10.0,
        max_red_time_s=90.0,
        backend="libsumo",
    )
    try:
        return run_episode(env, policy, seed=seed)
    finally:
        env.close()


scenario_results = {}
fixed_total = 0.0
print("Fixed-time baselines:")
for scenario_name, sumocfg_path in SCENARIOS.items():
    fixed_metrics = [
        eval_policy(sumocfg_path, FixedTimePolicy(green_duration_s=20.0), seed)
        for seed in EVAL_SEEDS
    ]
    agg = aggregate(fixed_metrics)
    fixed_score = score(agg)
    fixed_total += fixed_score
    print(scenario_name, agg, "score=", fixed_score)

results = []
for path in candidate_paths:
    total_score = 0.0
    details = {}
    trained_episode = None
    for scenario_name, sumocfg_path in SCENARIOS.items():
        agent = DQNAgent()
        trained_episode = load_checkpoint(path, agent)
        policy = DQNPolicy(agent, epsilon=0.0)
        metrics = [eval_policy(sumocfg_path, policy, seed) for seed in EVAL_SEEDS]
        agg = aggregate(metrics)
        scenario_score = score(agg)
        total_score += scenario_score
        details[scenario_name] = agg
    results.append((total_score, path, trained_episode, details))
    print(f"\n{path.name:<22} ep={trained_episode:<5} total_score={total_score:10.2f}")
    for scenario_name, agg in details.items():
        print(
            f"  {scenario_name:<16} mean_wait={agg['mean_waiting_time_s']:8.2f} "
            f"queue={agg['mean_queue_length']:6.2f} arrived={agg['arrived_vehicles']:7.2f}"
        )

results.sort(key=lambda row: row[0])
best_score, best_path, best_episode, best_details = results[0]
selected_path = CHECKPOINT_DIR / "dqn_eval_best.pt"
shutil.copy2(best_path, selected_path)
print("\nSelected:", best_path.name, "episode", best_episode, "score", best_score)
print("Wrote:", selected_path)


## 8. Download the selected model

Download `dqn_eval_best.pt` first. It is selected by deterministic evaluation across held-out seeds, so it is usually a better deployment candidate than the raw training `dqn_best.pt`. Keep `dqn_best.pt` and `dqn_final.pt` too if you want to compare locally.

In [ ]:
!ls -lh /kaggle/working/checkpoints


In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML, display


def download_link(path, filename=None):
    filename = filename or path.split("/")[-1]
    with open(path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return HTML(f'<a download="{filename}" href="data:application/octet-stream;base64,{b64}">Download {filename}</a>')


for path in [
    "/kaggle/working/checkpoints/dqn_eval_best.pt",
    "/kaggle/working/checkpoints/dqn_best.pt",
    "/kaggle/working/checkpoints/dqn_final.pt",
]:
    if Path(path).exists():
        display(download_link(path))
